In [8]:
from ragas.testset import TestsetGenerator
from langchain_core.documents import Document
from openai import OpenAI
from ragas.llms import llm_factory
import os
import anyio
from langchain_huggingface import HuggingFaceEmbeddings
import asyncio
import nest_asyncio
from ragas.run_config import RunConfig
import json



In [9]:
import sys
import types

# Add structural stubs to prevent Ragas from crashing on legacy LangChain VertexAI imports
if "langchain_community" not in sys.modules:
    sys.modules["langchain_community"] = types.ModuleType("langchain_community")
if "langchain_community.chat_models" not in sys.modules:
    sys.modules["langchain_community.chat_models"] = types.ModuleType("langchain_community.chat_models")
if "langchain_community.llms" not in sys.modules:
    sys.modules["langchain_community.llms"] = types.ModuleType("langchain_community.llms")

dummy_chat_module = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat_module.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat_module
sys.modules["langchain_community.llms"].VertexAI = type("VertexAI", (object,), {})

from ragas.testset import TestsetGenerator
from langchain_core.documents import Document
from openai import OpenAI
from ragas.llms import llm_factory
import os
import anyio
from ragas.embeddings import HuggingFaceEmbeddings
import asyncio
import nest_asyncio
from ragas.run_config import RunConfig
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

nest_asyncio.apply()
config = RunConfig(max_workers=2)
load_dotenv()

try:
    loop = asyncio.get_running_loop()
except RuntimeError:
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

embedding_model = HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

client = OpenAI(
    api_key="ollama",  # Ollama doesn't require a real key
    base_url="http://localhost:11434/v1"
)

llm = ChatOpenAI(
    model="gpt-oss:20b",
    api_key="ollama",
    base_url="http://localhost:11434/v1",
    max_tokens=4096,
)


documents = []
with open("arxiv_papers/articles.json", "r", encoding="utf-8") as file:
    data = json.load(file)

for chunk in data[:10]:
    print(chunk)
    metadata = {
        "article_id": chunk["article_id"],
        "title": chunk["title"]
    }
    documents.append(Document(page_content=chunk["abstract"], metadata=metadata))

generator = TestsetGenerator.from_langchain(
    llm=llm,
    embedding_model=embedding_model
)

testset = generator.generate_with_langchain_docs(
    documents=documents,
    testset_size=5,
    with_debugging_logs=True,
    run_config=config
)

testset_df = testset.to_pandas()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6920.19it/s]


{'article_id': '2609.31619v1', 'title': 'Learning to Stop without Learning to Stop: Self-Supervised Confidence Training Improves Reasoning Efficiency', 'abstract': "Reasoning models often generate very long reasoning traces, making inference computationally expensive. Existing approaches typically improve efficiency either through inference-time early-stopping mechanisms or by explicitly encouraging shorter reasoning during training, for example through reinforcement learning with length penalties. We show that substantial efficiency gains can instead emerge from a different kind of supervision: \\textit{confidence}. Using a self-supervised procedure, we fine-tune reasoning models to predict their confidence in the answer at intermediate points along their own reasoning trajectories using only 600 training problems. Confidence is used only as a training target: the loss contains no objective for reasoning length, efficiency, or stopping. At inference, the fine-tuned models use the stan

Applying EmbeddingExtractor:  60%|██████    | 6/10 [06:52<04:35, 68.77s/it]


KeyboardInterrupt: 